In [1]:
# ============================================================
# EXPERIMENT 1 — YOLOv8n BASELINE
# Load trained best model
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch

BEST_MODEL = Path(
    r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"
)

print("Model exists:", BEST_MODEL.exists())
print("Model path  :", BEST_MODEL)

model = YOLO(str(BEST_MODEL))

print("\nModel loaded successfully.")

Model exists: True
Model path  : G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt

Model loaded successfully.


In [2]:
# ============================================================
# MODEL INFORMATION
# ============================================================

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))

print("\nModel parameters:", sum(p.numel() for p in model.model.parameters()))

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU

Model parameters: 3011628


In [3]:
# ============================================================
# EXPERIMENT 1 — OFFICIAL TEST SET
# ============================================================

TEST_RESULTS = model.val(
    data=r"G:\EcoBotX_YOLO_FINAL\dataset.yaml",
    split="test",

    imgsz=640,
    batch=8,

    device=0,

    workers=4,

    plots=True,
    save_json=True,

    project=r"G:\EcoBotX_YOLO_training",
    name="experiment1_yolov8n_test"
)

print("\n============================================================")
print("EXPERIMENT 1 — YOLOv8n TEST RESULTS")
print("============================================================")

print("mAP50     :", TEST_RESULTS.box.map50)
print("mAP50-95  :", TEST_RESULTS.box.map)
print("Precision :", TEST_RESULTS.box.mp)
print("Recall    :", TEST_RESULTS.box.mr)

Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.1 ms, read: 0.80.3 MB/s, size: 6.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning G:\EcoBotX_YOLO_FINAL\labels\test... 1115 images, 216 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1115/1115 380.9it/s 2.9s0.0s
val: New cache created: G:\EcoBotX_YOLO_FINAL\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 12.0it/s 11.7s0.2s
                   all       1115        899      0.955      0.967      0.979      0.633
                BOTTLE        217        217      0.974      0.982      0.987      0.579
                   CAN        228        228      0.906      0.908   

In [4]:
# ============================================================
# PER-CLASS TEST PERFORMANCE
# ============================================================

class_names = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

print("\n============================================================")
print("PER-CLASS TEST RESULTS")
print("============================================================")

for i, name in class_names.items():

    print(
        f"{name:10s} | "
        f"Precision: {TEST_RESULTS.box.p[i]:.4f} | "
        f"Recall: {TEST_RESULTS.box.r[i]:.4f} | "
        f"mAP50: {TEST_RESULTS.box.ap50[i]:.4f} | "
        f"mAP50-95: {TEST_RESULTS.box.ap[i]:.4f}"
    )


PER-CLASS TEST RESULTS
BOTTLE     | Precision: 0.9744 | Recall: 0.9816 | mAP50: 0.9867 | mAP50-95: 0.5788
CAN        | Precision: 0.9064 | Recall: 0.9079 | mAP50: 0.9471 | mAP50-95: 0.6063
PAPER      | Precision: 0.9558 | Recall: 0.9920 | mAP50: 0.9913 | mAP50-95: 0.6812
WRAPPER    | Precision: 0.9853 | Recall: 0.9853 | mAP50: 0.9907 | mAP50-95: 0.6677


In [5]:
# ============================================================
# SAVE EXPERIMENT 1 BASELINE
# ============================================================

import json

baseline = {
    "model": "YOLOv8n",
    "experiment": "Experiment 1",
    "image_size": 640,
    "parameters": 3006428,
    "gflops": 8.1,

    "test_precision": float(TEST_RESULTS.box.mp),
    "test_recall": float(TEST_RESULTS.box.mr),
    "test_mAP50": float(TEST_RESULTS.box.map50),
    "test_mAP50_95": float(TEST_RESULTS.box.map),

    "classes": {
        "BOTTLE": {
            "precision": float(TEST_RESULTS.box.p[0]),
            "recall": float(TEST_RESULTS.box.r[0]),
            "mAP50": float(TEST_RESULTS.box.ap50[0]),
            "mAP50_95": float(TEST_RESULTS.box.ap[0])
        },

        "CAN": {
            "precision": float(TEST_RESULTS.box.p[1]),
            "recall": float(TEST_RESULTS.box.r[1]),
            "mAP50": float(TEST_RESULTS.box.ap50[1]),
            "mAP50_95": float(TEST_RESULTS.box.ap[1])
        },

        "PAPER": {
            "precision": float(TEST_RESULTS.box.p[2]),
            "recall": float(TEST_RESULTS.box.r[2]),
            "mAP50": float(TEST_RESULTS.box.ap50[2]),
            "mAP50_95": float(TEST_RESULTS.box.ap[2])
        },

        "WRAPPER": {
            "precision": float(TEST_RESULTS.box.p[3]),
            "recall": float(TEST_RESULTS.box.r[3]),
            "mAP50": float(TEST_RESULTS.box.ap50[3]),
            "mAP50_95": float(TEST_RESULTS.box.ap[3])
        }
    }
}

BASELINE_FILE = Path(
    r"G:\EcoBotX_YOLO_training\experiment1_baseline.json"
)

with open(BASELINE_FILE, "w") as f:
    json.dump(baseline, f, indent=4)

print("Baseline saved to:")
print(BASELINE_FILE)

Baseline saved to:
G:\EcoBotX_YOLO_training\experiment1_baseline.json


In [6]:
# ============================================================
# BASELINE SUMMARY
# ============================================================

print("\n")
print("=" * 65)
print("              EXPERIMENT 1 — YOLOv8n")
print("=" * 65)

print(f"Parameters       : {baseline['parameters']:,}")
print(f"GFLOPs           : {baseline['gflops']}")
print(f"Test Precision   : {baseline['test_precision']:.4f}")
print(f"Test Recall      : {baseline['test_recall']:.4f}")
print(f"Test mAP50       : {baseline['test_mAP50']:.4f}")
print(f"Test mAP50-95    : {baseline['test_mAP50_95']:.4f}")

print("=" * 65)



              EXPERIMENT 1 — YOLOv8n
Parameters       : 3,006,428
GFLOPs           : 8.1
Test Precision   : 0.9555
Test Recall      : 0.9667
Test mAP50       : 0.9790
Test mAP50-95    : 0.6335
